In [21]:
import os
import sys
module_path = os.path.abspath(os.path.join('..', '..', '..', '..'))
sys.path.insert(0, module_path)

### Подготовка
1. Загрузим данные
2. Выкинем implicit feedback в виде нулей
3. Time-split на train и test. 90% данных оставим на train, 10% на тест

In [22]:

from brs.models.rate_prediction.baseline.model import BaselineRateModel
from brs.ingestion import LoadLocalDataset

lld = LoadLocalDataset()
data_books = lld.load_dataset("books", clear_version=True)
data_users = lld.load_dataset("users", clear_version=True)
data_ratings = lld.load_dataset("ratings", clear_version=True)

In [23]:
from brs.models.rate_prediction.lin_reg.pipeline import RatingsModelPipeline, UsersModelPipeline
data_ratings = RatingsModelPipeline().fit_transform(data_ratings)
data_users = UsersModelPipeline(ratings_df=data_ratings).fit_transform(data_users)

In [24]:
num_ratings = data_ratings.shape[0]
split = round(num_ratings * 0.9)
train, test = data_ratings[:split], data_ratings[split:]

### Pipeline/preprocess

- Считаем среднюю оценку для каждой книги на train
- Предсказываем каждому пользователю на test именно её

In [25]:
from brs.preprocessing.pipeline import NaPreprocessing, DummiesPreprocessing, TopKDummiesPreprocessing, DropPreprocessing
from brs.pipelines.dataframe import DataframePipeline
from brs.feature_engineering.pipeline import UserSetAgeFeature, UserSetCountryFeature, UserSetNumRatingFeature, UserSetMeanRatingFeature

Na_dict = {
    "Age": {"strategy": "mean"},
    "Country": {"strategy": "most_frequent"}, 
}

users_pipeline = DataframePipeline([
    UserSetCountryFeature(),
    NaPreprocessing({
        "Age": {"strategy": "mean"},
        "Country": {"strategy": "most_frequent"}, 
    }),
    UserSetAgeFeature(),
    UserSetNumRatingFeature(ratings_df=train),
    UserSetMeanRatingFeature(ratings_df=train),
    NaPreprocessing({
        "Num-Ratings": {"strategy": "mean"},
        "Mean-Rating": {"strategy": "mean"}
    }),
    TopKDummiesPreprocessing(columns=["Country"], top_k=15, drop_first=True),
    DummiesPreprocessing(columns=["Age-Group"]),
    DropPreprocessing(columns=["Location"])
])

data_users = users_pipeline.fit_transform(data_users)

In [26]:
from brs.preprocessing.pipeline import BooksClearifyPublicationYear, BooksSetDefaultTypes
from brs.feature_engineering.pipeline import BookSetElderityFeature, BookSetPopularityFeature, BookSetRatingFeature

books_pipeline = DataframePipeline([
    BooksSetDefaultTypes(),
    NaPreprocessing({
        "Year-Of-Publication": {"strategy": "mean"},
        "Book-Author": {"strategy": "constant", "const_value": "unknown"},
        "Publisher": {"strategy": "constant", "const_value": "unknown"},
    }),
    BooksClearifyPublicationYear(),
    BookSetElderityFeature(),
    BookSetPopularityFeature(ratings_df=train, log=False),
    BookSetRatingFeature(ratings_df=train),
    NaPreprocessing({
        "Popularity": {"strategy": "mean"},
        "Mean-Rating": {"strategy": "mean"},
    }),
    TopKDummiesPreprocessing(columns=["Publisher", "Book-Author"], top_k=15, drop_first=True),
    DropPreprocessing(columns=["Book-Title"])
])

In [27]:
data_books = books_pipeline.fit_transform(data_books)

In [28]:
data_books.head()

,ISBN,Year-Of-Publication,Elderity,Elderity_log,Popularity,Mean-Rating,Bantam Books,Berkley Publishing Group,Fawcett Books,Harlequin,...,Isaac Asimov,Mark Twain,Nora Roberts,Not Applicable (Na ),Other,Piers Anthony,R. L. Stine,Stephen King,Terry Pratchett,William Shakespeare
0,0195153448,2002,21,3.091042,1.905341,7.409485,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1,0002005018,2001,22,3.135494,3.000000,7.333333,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,0060973129,1991,32,3.496508,2.000000,7.500000,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,0374157065,1999,24,3.218876,3.000000,6.666667,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
4,0393045218,1999,24,3.218876,1.905341,7.409485,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


In [29]:
train = (train.merge(data_users, how="inner", on="User-ID")
         .merge(data_books, how="inner", on="ISBN")
         .drop(columns=["User-ID", "ISBN"], axis=1))
train_X, train_y = train.drop(columns=["Book-Rating"], axis=1), train["Book-Rating"]

test = (test.merge(data_users, how="inner", on="User-ID")
         .merge(data_books, how="inner", on="ISBN")
         .drop(columns=["User-ID", "ISBN"], axis=1))
test_X, test_y = test.drop(columns=["Book-Rating"], axis=1), test["Book-Rating"]


In [30]:
from brs.models.rate_prediction.lin_reg import LinRegModel

model = LinRegModel()
model.fit(train_X, train_y)

predictions = model.predict(test_X)


### Считаем качество наших предсказаний

Три ключевые метрики лишь немного лучше бэйзлайна по (MSE и RMSE)

In [31]:
from brs.metrics.regression_metrics import MAE, RMSE, MSE

mae_score = MAE().estimate(predictions, test_y)
rmse_score = RMSE().estimate(predictions, test_y)
mse_score = MSE().estimate(predictions, test_y)

print(f"MAE: {mae_score}")
print(f"RMSE: {rmse_score}")
print(f"MSE: {mse_score}")

MAE: 1.568399179909841
RMSE: 1.955431680922467
MSE: 3.8237130587552643
